In [2]:
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Bhushan\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


In [3]:
%pip install kagglehub



   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   -------------------- ------------------- 1/2 [kagglehub]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
from fredapi import Fred
from sklearn.preprocessing import StandardScaler

In [4]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import sys
print(sys.executable)

C:\Users\Bhushan\anaconda3\python.exe


In [6]:
!C:\Users\Bhushan\anaconda3\python.exe -m pip install fredapi


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df = pd.read_csv("telco.csv")

In [5]:
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())

Shape: (7043, 21)

Missing values per column:
 customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [6]:
blank_mask = df["TotalCharges"].astype(str).str.strip() == ""
print("\nBlank TotalCharges rows:", blank_mask.sum())
print("Their tenure values:", df.loc[blank_mask, "tenure"].unique())


Blank TotalCharges rows: 11
Their tenure values: [0]


In [7]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [8]:
df.loc[df["tenure"] == 0, "TotalCharges"] = 0

In [9]:
df = df.drop(columns=["customerID"])

In [10]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [11]:
assert df.isnull().sum().sum() == 0

In [12]:
print("\nAfter cleaning — nulls remaining:", df.isnull().sum().sum())
print("Churn rate:", df["Churn"].mean().round(3))


After cleaning — nulls remaining: 0
Churn rate: 0.265


In [13]:
OBSERVATION_DATE = pd.Timestamp("2024-01-01")
df["signup_date"] = OBSERVATION_DATE - pd.to_timedelta(df["tenure"] * 30, unit="D")

In [14]:
df = df.sort_values("signup_date").reset_index(drop=True)

In [15]:
N_WINDOWS = 10
df["window_id"] = pd.qcut(df["signup_date"].rank(method="first"), N_WINDOWS, labels=False)

In [16]:
window_churn_diagnostic = df.groupby("window_id")["Churn"].agg(
    churn_rate="mean", count="size"
)
print("\n--- Per-window churn rate (diagnostic) ---")
print(window_churn_diagnostic)

# Save it separately so it's easy to pull straight into the report as a table
window_churn_diagnostic.to_csv("window_churn_rate_diagnostic.csv")


--- Per-window churn rate (diagnostic) ---
           churn_rate  count
window_id                   
0            0.041135    705
1            0.092330    704
2            0.134943    704
3            0.178977    704
4            0.218440    705
5            0.232955    704
6            0.315341    704
7            0.367898    704
8            0.463068    704
9            0.608511    705


In [17]:
print("\nCustomers per window:\n", df.groupby("window_id").size())


Customers per window:
 window_id
0    705
1    704
2    704
3    704
4    705
5    704
6    704
7    704
8    704
9    705
dtype: int64


In [18]:
FRED_API_KEY = "197deb9fbed339326ccd144adcd13ba0"

In [19]:
try:
    fred = Fred(api_key=FRED_API_KEY)
    cpi_raw = fred.get_series("INDCPIALLMINMEI")  
    cpi = cpi_raw.rename("india_cpi").to_frame()
    cpi.index.name = "date"
    cpi = cpi.reset_index()
except Exception as e:
    print(f"\n[WARNING] Could not fetch FRED data ({e}). Using placeholder CPI.")
    dates = pd.date_range("2000-01-01", "2024-12-01", freq="MS")
    cpi = pd.DataFrame({"date": dates, "india_cpi": 100.0})

In [20]:
cpi["year_month"] = cpi["date"].dt.to_period("M")
df["year_month"] = df["signup_date"].dt.to_period("M")

In [21]:
df = df.merge(cpi[["year_month", "india_cpi"]], on="year_month", how="left")
 
missing_cpi = df["india_cpi"].isnull().sum()
print(f"\nRows with no matching CPI value: {missing_cpi}")


Rows with no matching CPI value: 0


In [22]:
df["india_cpi"] = df["india_cpi"].ffill()

In [23]:
cpi["cpi_mom_pct_change"] = cpi["india_cpi"].pct_change() * 100
df = df.merge(cpi[["year_month", "cpi_mom_pct_change"]], on="year_month", how="left")
df["cpi_mom_pct_change"] = df["cpi_mom_pct_change"].fillna(0)

In [24]:
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

In [25]:
multi_cat_cols = [
    "gender", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaymentMethod"
]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

In [26]:
print("\nColumns after encoding:", df.shape[1])


Columns after encoding: 36


In [27]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "india_cpi", "cpi_mom_pct_change"]

In [28]:
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

In [29]:
def get_rolling_splits(data, window_col="window_id"):
    windows = sorted(data[window_col].unique())
    splits = []
    for i in range(len(windows) - 1):
        train = data[data[window_col] == windows[i]].copy()
        test = data[data[window_col] == windows[i + 1]].copy()
        splits.append((train, test))
    return splits
 
splits = get_rolling_splits(df)
print(f"\nBuilt {len(splits)} rolling train/test window pairs")


Built 9 rolling train/test window pairs


In [30]:
leak_cols = ["signup_date", "year_month", "window_id"]
model_features = [c for c in df.columns if c not in leak_cols + ["Churn"]]
 
print("\n--- Final checks ---")
print("Churn rate (sanity check, expect ~26-27% on real data):", df["Churn"].mean().round(3))
print("Any nulls in model features?:", df[model_features].isnull().sum().sum())
print("Total model features:", len(model_features))
print("Window sizes:\n", df.groupby("window_id").size())
 
print("\nPhase 1 complete. `df` and `splits` are ready for Phase 2.")


--- Final checks ---
Churn rate (sanity check, expect ~26-27% on real data): 0.265
Any nulls in model features?: 0
Total model features: 32
Window sizes:
 window_id
0    705
1    704
2    704
3    704
4    705
5    704
6    704
7    704
8    704
9    705
dtype: int64

Phase 1 complete. `df` and `splits` are ready for Phase 2.


In [31]:
print(df[["tenure", "MonthlyCharges", "TotalCharges", "india_cpi", "cpi_mom_pct_change"]].describe())

            tenure  MonthlyCharges  TotalCharges    india_cpi  \
count  7043.000000     7043.000000   7043.000000  7043.000000   
mean     32.371149       64.761692   2279.734304   134.153897   
std      24.559481       30.090047   2266.794470    15.077554   
min       0.000000       18.250000      0.000000   109.786400   
25%       9.000000       35.500000    398.550000   120.879800   
50%      29.000000       70.350000   1394.550000   135.516500   
75%      55.000000       89.850000   3786.600000   147.856200   
max      72.000000      118.750000   8684.800000   153.915900   

       cpi_mom_pct_change  
count         7043.000000  
mean             0.404920  
std              0.800638  
min             -1.221270  
25%             -0.215654  
50%              0.372592  
75%              0.675180  
max              3.436508  


In [32]:
df.head(10)

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,signup_date,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,1,72,1,0,25.10,1857.85,0,2018-02-01,...,0,1,0,1,0,0,1,0,0,0
1,0,1,0,72,1,1,86.40,6058.95,0,2018-02-01,...,1,0,1,0,1,0,1,0,0,0
2,0,1,1,72,1,0,79.05,5730.70,0,2018-02-01,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,72,1,1,112.75,8192.60,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0
4,0,1,0,72,1,1,108.95,7875.00,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0
5,0,1,1,72,1,0,105.25,7609.75,0,2018-02-01,...,0,0,1,0,1,0,1,0,0,0
6,1,1,0,72,1,1,89.85,6697.35,0,2018-02-01,...,1,0,1,0,1,0,1,0,0,0
7,0,1,1,72,1,1,24.05,1709.15,0,2018-02-01,...,0,1,0,1,0,0,1,0,0,0
8,0,0,0,72,1,1,84.70,6185.15,0,2018-02-01,...,1,0,0,0,0,0,1,0,0,0
9,1,1,0,72,1,1,114.65,8333.95,0,2018-02-01,...,1,0,1,0,1,0,1,0,0,0


In [33]:
print(df.shape)          
print(df.columns.tolist())

(7043, 36)
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'signup_date', 'window_id', 'year_month', 'india_cpi', 'cpi_mom_pct_change', 'gender_Male', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [34]:
df["tenure"]              
df[["tenure", "Churn"]]

,tenure,Churn
0,72,0
1,72,0
2,72,0
3,72,0
4,72,0
...,...,...
7038,0,0
7039,0,0
7040,0,0
7041,0,0


In [35]:
df.describe()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,signup_date,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043,...,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,0.483033,0.299588,32.371149,0.903166,0.592219,64.761692,2279.734304,0.265370,2021-05-04 20:46:22.677836032,...,0.290217,0.216669,0.384353,0.216669,0.387903,0.209144,0.240664,0.216101,0.335794,0.228880
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.250000,0.000000,0.000000,2018-02-01 00:00:00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,9.000000,1.000000,0.000000,35.500000,398.550000,0.000000,2019-06-26 00:00:00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,29.000000,1.000000,1.000000,70.350000,1394.550000,0.000000,2021-08-14 00:00:00,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,1.000000,1.000000,55.000000,1.000000,1.000000,89.850000,3786.600000,1.000000,2023-04-06 00:00:00,...,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,72.000000,1.000000,1.000000,118.750000,8684.800000,1.000000,2024-01-01 00:00:00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
std,0.368612,0.499748,0.458110,24.559481,0.295752,0.491457,30.090047,2266.794470,0.441561,NaN,...,0.453895,0.412004,0.486477,0.412004,0.487307,0.406726,0.427517,0.411613,0.472301,0.420141


In [36]:
df[["tenure", "signup_date", "window_id", "india_cpi", "cpi_mom_pct_change"]].head(15)

,tenure,signup_date,window_id,india_cpi,cpi_mom_pct_change
0,72,2018-02-01,0,109.7864,-0.347285
1,72,2018-02-01,0,109.7864,-0.347285
2,72,2018-02-01,0,109.7864,-0.347285
3,72,2018-02-01,0,109.7864,-0.347285
4,72,2018-02-01,0,109.7864,-0.347285
5,72,2018-02-01,0,109.7864,-0.347285
6,72,2018-02-01,0,109.7864,-0.347285
7,72,2018-02-01,0,109.7864,-0.347285
8,72,2018-02-01,0,109.7864,-0.347285
9,72,2018-02-01,0,109.7864,-0.347285


In [37]:
print(len(splits))            # how many (train, test) pairs you have — should be 9
train0, test0 = splits[0]     # unpack the very first pair
print(train0.shape, test0.shape)
train0.head()

9
(705, 36) (704, 36)


,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,signup_date,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,1,72,1,0,25.10,1857.85,0,2018-02-01,...,0,1,0,1,0,0,1,0,0,0
1,0,1,0,72,1,1,86.40,6058.95,0,2018-02-01,...,1,0,1,0,1,0,1,0,0,0
2,0,1,1,72,1,0,79.05,5730.70,0,2018-02-01,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,72,1,1,112.75,8192.60,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0
4,0,1,0,72,1,1,108.95,7875.00,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0


In [38]:
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,signup_date,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,1,72,1,0,25.10,1857.85,0,2018-02-01,...,0,1,0,1,0,0,1,0,0,0
1,0,1,0,72,1,1,86.40,6058.95,0,2018-02-01,...,1,0,1,0,1,0,1,0,0,0
2,0,1,1,72,1,0,79.05,5730.70,0,2018-02-01,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,72,1,1,112.75,8192.60,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0
4,0,1,0,72,1,1,108.95,7875.00,0,2018-02-01,...,1,0,1,0,1,0,1,1,0,0


In [39]:
df.groupby("window_id").size()

window_id
0    705
1    704
2    704
3    704
4    705
5    704
6    704
7    704
8    704
9    705
dtype: int64

In [40]:
df.to_csv('churnData.csv', index=False)